# Predilex NLP — Entraînement sur Kaggle

Ce notebook entraîne les deux modèles CamemBERT sur GPU Kaggle.

**Avant de commencer :**
1. `Settings` (à droite) → `Accelerator` → **GPU T4 x2**
2. `Settings` → `Internet` → **On** (pour cloner GitHub)
3. Avoir les données uploadées sur Kaggle (voir Étape 3)

**Durée estimée :**
- GPU T4 : ~1h (sexe ~15 min + dates ~45 min)

---
## Étape 1 — Vérification du GPU

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU détecté : {gpu_name} ({gpu_mem:.1f} GB VRAM)")
else:
    print("❌ Pas de GPU — active le GPU : Settings > Accelerator > GPU T4")
    raise SystemExit("GPU requis")

---
## Étape 2 — Installation des dépendances

In [ ]:
!pip install -q transformers[torch] accelerate sentencepiece datasets pyyaml scikit-learn
print("✅ Dépendances installées")

import transformers, torch, accelerate
print(f"  torch        : {torch.__version__}")
print(f"  transformers : {transformers.__version__}")
print(f"  accelerate   : {accelerate.__version__}")

---
## Étape 3 — Cloner le repo GitHub

Si repo **privé** : génère un token sur https://github.com/settings/tokens (scope : repo)

In [ ]:
import os
from getpass import getpass

TOKEN = getpass("GitHub token (laisser vide si repo public) : ")

if TOKEN:
    GITHUB_URL = f"https://{TOKEN}@github.com/djema-rayane/predilex-nlp.git"
else:
    GITHUB_URL = "https://github.com/djema-rayane/predilex-nlp.git"

REPO_DIR = "/kaggle/working/predilex-nlp"

if os.path.exists(REPO_DIR):
    print("Repo déjà cloné — mise à jour...")
    os.chdir(REPO_DIR)
    !git pull
else:
    !git clone {GITHUB_URL} {REPO_DIR}
    os.chdir(REPO_DIR)

print(f"✅ Répertoire de travail : {os.getcwd()}")
!ls

---
## Étape 4 — Upload et lien des données

**Comment uploader tes données sur Kaggle :**
1. Va sur https://www.kaggle.com/datasets
2. `New Dataset` → uploade un zip contenant :
   - `Y_train_predilex.csv`
   - `x_train_ids.csv`
   - `txt_files/` (dossier avec les 770 .txt)
3. Dans ce notebook → `Add data` (icône + en haut à droite) → cherche ton dataset
4. Il sera disponible dans `/kaggle/input/[nom-du-dataset]/`

**Modifie `KAGGLE_DATA_DIR` ci-dessous avec le bon chemin :**

In [ ]:
import os
from pathlib import Path

# ⚠️ Modifie ce chemin avec le nom de ton dataset Kaggle
KAGGLE_DATA_DIR = "/kaggle/input/predilex-data"  # ← à adapter

# Vérification
required = [
    f"{KAGGLE_DATA_DIR}/Y_train_predilex.csv",
    f"{KAGGLE_DATA_DIR}/x_train_ids.csv",
    f"{KAGGLE_DATA_DIR}/txt_files",
]
for path in required:
    exists = os.path.exists(path)
    status = "✅" if exists else "❌"
    print(f"{status} {path}")
    if not exists:
        raise FileNotFoundError(f"Fichier manquant : {path} — vérifie KAGGLE_DATA_DIR")

# Créer les dossiers dans le projet
REPO_DIR = "/kaggle/working/predilex-nlp"
RAW = f"{REPO_DIR}/data/raw"
Path(f"{REPO_DIR}/data/raw").mkdir(parents=True, exist_ok=True)
Path(f"{REPO_DIR}/data/processed").mkdir(parents=True, exist_ok=True)
Path(f"{REPO_DIR}/models").mkdir(parents=True, exist_ok=True)

# Liens symboliques (évite de copier 770 fichiers)
for fname in ["Y_train_predilex.csv", "x_train_ids.csv"]:
    dst = f"{RAW}/{fname}"
    if not os.path.exists(dst):
        os.symlink(f"{KAGGLE_DATA_DIR}/{fname}", dst)

txt_dst = f"{RAW}/txt_files"
if not os.path.exists(txt_dst):
    os.symlink(f"{KAGGLE_DATA_DIR}/txt_files", txt_dst)

n_txt = len(list(Path(txt_dst).glob("*.txt")))
print(f"\n✅ Données liées : {n_txt} fichiers .txt")

---
## Étape 5 — Vérification du chargement des données

In [ ]:
import sys
REPO_DIR = "/kaggle/working/predilex-nlp"
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

from src.data_loader import load_config, load_train_data

cfg = load_config('configs/config.yaml')
df  = load_train_data(cfg)

print(f"\n✅ {len(df)} documents chargés")
print(f"Sexe      : {df['sexe'].value_counts().to_dict()}")
print(f"Text moy  : {df['text'].str.len().mean():.0f} chars")

---
## Étape 6 — Entraînement du modèle SEXE

Fine-tuning CamemBERT pour prédire homme/femme.
- **Input** : premiers 512 tokens du document
- **Classes** : 2 (homme=0, femme=1)
- **Durée** : ~15 min sur T4

In [ ]:
!python src/train_sex.py --config configs/config.yaml

In [ ]:
from pathlib import Path
sex_model_path = Path('models/sex_classifier/best_model')
if sex_model_path.exists():
    files = list(sex_model_path.iterdir())
    print(f"✅ Modèle sexe sauvegardé : {len(files)} fichiers")
    for f in files:
        size = f.stat().st_size / 1e6
        print(f"   {f.name:<35} {size:.1f} MB")
else:
    print("❌ Modèle introuvable — vérifier les logs ci-dessus")

---
## Étape 7 — Entraînement du modèle DATES

Fine-tuning CamemBERT pour classifier les phrases de dates.
- **Input** : phrase + contexte ±2 (256 tokens)
- **Classes** : 3 (date_accident=0, date_consolidation=1, autre_date=2)
- **Durée** : ~45 min sur T4

In [ ]:
!python src/train_dates.py --config configs/config.yaml

In [ ]:
date_model_path = Path('models/date_classifier/best_model')
if date_model_path.exists():
    files = list(date_model_path.iterdir())
    print(f"✅ Modèle dates sauvegardé : {len(files)} fichiers")
    for f in files:
        size = f.stat().st_size / 1e6
        print(f"   {f.name:<35} {size:.1f} MB")
else:
    print("❌ Modèle introuvable — vérifier les logs ci-dessus")

---
## Étape 8 — Évaluer sans réentraîner (si erreur en cours de run)

Si le modèle est sauvegardé mais les métriques n'ont pas été affichées, lance cette cellule.

In [ ]:
!python src/eval_saved_model.py --config configs/config.yaml

---
## Étape 9 — Analyse des erreurs

In [ ]:
# Analyse complète des erreurs
!python src/analyze_errors.py --config configs/config.yaml

# Pour voir plus d'exemples :
# !python src/analyze_errors.py --config configs/config.yaml --task consolidation --n 5

---
## Étape 10 — Sauvegarder les modèles

> ⚠️ **IMPORTANT** : Kaggle supprime les fichiers à la fin de la session.
> Utilise le bouton **Save Version** en haut à droite pour sauvegarder les outputs,
> ou télécharge les modèles manuellement.

In [ ]:
import shutil
from pathlib import Path

# Les outputs Kaggle sont persistés dans /kaggle/working/
# Copier les modèles dans un dossier output propre
OUTPUT_DIR = "/kaggle/working/predilex_models"
os.makedirs(OUTPUT_DIR, exist_ok=True)

for model_name in ["sex_classifier", "date_classifier"]:
    src = f"/kaggle/working/predilex-nlp/models/{model_name}/best_model"
    dst = f"{OUTPUT_DIR}/{model_name}"

    if os.path.exists(src):
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        size_mb = sum(f.stat().st_size for f in Path(dst).rglob('*') if f.is_file()) / 1e6
        print(f"✅ {model_name} copié ({size_mb:.0f} MB)")
    else:
        print(f"❌ {model_name} introuvable")

print(f"\n✅ Modèles dans : {OUTPUT_DIR}")
print("Clique sur 'Save Version' en haut à droite pour persister les outputs Kaggle.")

---
## Étape 11 — Télécharger les modèles

Pour télécharger les modèles sur ta machine locale :

In [ ]:
import zipfile
from pathlib import Path

# Zipper les modèles pour téléchargement
zip_path = "/kaggle/working/predilex_models.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for model_name in ["sex_classifier", "date_classifier"]:
        src = Path(f"/kaggle/working/predilex_models/{model_name}")
        if src.exists():
            for f in src.rglob("*"):
                if f.is_file():
                    zf.write(f, f.relative_to("/kaggle/working/predilex_models"))

size_mb = Path(zip_path).stat().st_size / 1e6
print(f"✅ Archive créée : {zip_path} ({size_mb:.0f} MB)")
print("Télécharge le fichier depuis le panneau de fichiers à droite (icône dossier).")

---
## Étape 12 — Inférence sur le test set (soumission)

> À lancer quand le test set Predilex est disponible.

In [ ]:
# Lier le test set (quand disponible)
# TEST_DATA_DIR = "/kaggle/input/predilex-test"  # ← à adapter
# os.symlink(f"{TEST_DATA_DIR}/x_test_ids.csv", "data/raw/x_test_ids.csv")
# os.symlink(f"{TEST_DATA_DIR}/txt_files", "data/raw/x_test_txt_files")

# !python src/predict.py

print("Décommenter les lignes ci-dessus quand le test set est disponible.")